# U-Net 01 - Pré-processamento e Geração de Shards

**Objetivo:** Converter tiles de mosaico e GTv2 em shards NPZ para treino/validação/teste.

**Entradas esperadas:**
- Tiles `unet_mt_2023_mosaic_*`
- Tiles `unet_mt_2023_gtv2_*`

**Saídas geradas:**
- Shards em `/content/drive/MyDrive/unet_dataset_mt2023_v2/shards`
- Manifesto em `/content/drive/MyDrive/unet_dataset_mt2023_v2/logs/manifest_tiles.csv`

**Posição no fluxo:** Etapa 1 do fluxo canônico U-Net.


In [7]:
# CÉLULA 1 — mount do drive (reforçado)
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
assert os.path.exists("/content/drive/MyDrive"), "drive não montou corretamente"
print("drive mount OK ✅")

Mounted at /content/drive
drive mount OK ✅


In [3]:
# CÉLULA 2 — imports + checagens
from pathlib import Path
import re, json, time, shutil, gc
import numpy as np
import pandas as pd

import rasterio
from rasterio.windows import Window
from tqdm import tqdm

print("rasterio:", rasterio.__version__)
print("numpy:", np.__version__)

rasterio: 1.5.0
numpy: 2.0.2


In [4]:
# CÉLULA 3 — caminhos (1 pasta local só) + saída persistente no drive

# fonte (drive)
SRC_DRIVE = Path("/content/drive/MyDrive/GEE_Exports")
assert SRC_DRIVE.exists(), f"não achei: {SRC_DRIVE}"

# destino local único
LOCAL_ROOT = Path("/content/gee_exports_local")
LOCAL_MOSAIC = LOCAL_ROOT / "mosaic"
LOCAL_GT     = LOCAL_ROOT / "gt"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_MOSAIC.mkdir(parents=True, exist_ok=True)
LOCAL_GT.mkdir(parents=True, exist_ok=True)

# saída persistente no drive (dataset final para treinar unet)
OUT_DRIVE_ROOT = Path("/content/drive/MyDrive/unet_dataset_mt2023_v2")
OUT_SHARDS = OUT_DRIVE_ROOT / "shards"   # shards .npz
OUT_LOGS   = OUT_DRIVE_ROOT / "logs"     # manifests/stats
OUT_SHARDS.mkdir(parents=True, exist_ok=True)
OUT_LOGS.mkdir(parents=True, exist_ok=True)

print("src:", SRC_DRIVE)
print("local:", LOCAL_ROOT)
print("out:", OUT_DRIVE_ROOT)

src: /content/drive/MyDrive/GEE_Exports
local: /content/gee_exports_local
out: /content/drive/MyDrive/unet_dataset_mt2023_v2


In [5]:
# CÉLULA 4 — indexa mosaicos e gts no drive (sem api)

mosaic_drive = sorted(SRC_DRIVE.glob("unet_mt_2023_mosaic_x*_y*.tif"))
gt_drive     = sorted(SRC_DRIVE.glob("unet_mt_2023_gtv2_x*_y*.tif"))

print("drive mosaics:", len(mosaic_drive))
print("drive gts:", len(gt_drive))

assert len(mosaic_drive) == 169, "não achou 169 mosaics no drive"
assert len(gt_drive) == 169, "não achou 169 gts no drive"
print("index ok ✅")

drive mosaics: 169
drive gts: 169
index ok ✅


In [6]:
# CÉLULA 5 — cópia drive -> local com resume + retry + remount

from pathlib import Path
import time, shutil, os
from tqdm import tqdm
from google.colab import drive

SRC_DRIVE = Path("/content/drive/MyDrive/GEE_Exports")
LOCAL_ROOT = Path("/content/gee_exports_local")
LOCAL_MOSAIC = LOCAL_ROOT / "mosaic"
LOCAL_GT     = LOCAL_ROOT / "gt"
LOCAL_MOSAIC.mkdir(parents=True, exist_ok=True)
LOCAL_GT.mkdir(parents=True, exist_ok=True)

mosaic_drive = sorted(SRC_DRIVE.glob("unet_mt_2023_mosaic_x*_y*.tif"))
gt_drive     = sorted(SRC_DRIVE.glob("unet_mt_2023_gtv2_x*_y*.tif"))

def remount_drive():
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception:
        pass

def is_ok(src: Path, dst: Path) -> bool:
    try:
        return dst.exists() and dst.stat().st_size == src.stat().st_size and dst.stat().st_size > 0
    except Exception:
        return False

def safe_copy(src: Path, dst: Path, retries=6) -> bool:
    for i in range(retries):
        try:
            if is_ok(src, dst):
                return True
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            if is_ok(src, dst):
                return True
        except OSError as e:
            # drive caiu / transport endpoint / etc.
            time.sleep(2 * (i + 1))
            remount_drive()
        except Exception:
            time.sleep(2 * (i + 1))
            remount_drive()
    return False

def copy_list(files, dst_dir: Path, label: str, batch=25):
    pending = []
    for p in files:
        dst = dst_dir / p.name
        if not is_ok(p, dst):
            pending.append((p, dst))

    print(f"{label}: total={len(files)} ok={len(files)-len(pending)} pendente={len(pending)}")
    if not pending:
        return

    # copia em lotes
    for b0 in range(0, len(pending), batch):
        chunk = pending[b0:b0+batch]
        print(f"{label}: lote {b0}..{b0+len(chunk)-1} ({len(chunk)} arquivos)")
        for src, dst in tqdm(chunk, desc=f"copy {label}"):
            ok = safe_copy(src, dst)
            if not ok:
                raise RuntimeError(f"falhou ao copiar após retries: {src.name}")

        # pausa curta para reduzir chance de desconexão
        time.sleep(2)

# roda
copy_list(mosaic_drive, LOCAL_MOSAIC, "mosaic", batch=20)
copy_list(gt_drive,     LOCAL_GT,     "gt",     batch=20)

print("copy resume OK ✅")
print("local mosaics:", len(list(LOCAL_MOSAIC.glob('*.tif'))))
print("local gts:", len(list(LOCAL_GT.glob('*.tif'))))

mosaic: total=169 ok=12 pendente=157
mosaic: lote 0..19 (20 arquivos)


copy mosaic: 100%|██████████| 20/20 [01:52<00:00,  5.60s/it]


mosaic: lote 20..39 (20 arquivos)


copy mosaic: 100%|██████████| 20/20 [01:23<00:00,  4.15s/it]


mosaic: lote 40..59 (20 arquivos)


...

gt: lote 140..159 (20 arquivos)


copy gt: 100%|██████████| 20/20 [00:03<00:00,  5.60it/s]


gt: lote 160..168 (9 arquivos)


copy gt: 100%|██████████| 9/9 [00:01<00:00,  5.42it/s]


copy resume OK ✅
local mosaics: 169
local gts: 169


In [9]:
# CÉLULA 6 — manifest (pareamento mosaic <-> gt por key x/y do filename)

rx = re.compile(r"unet_mt_2023_(mosaic|gtv2)_x(?P<x>[-0-9p]+)_y(?P<y>[-0-9p]+)\.tif$", re.I)

def tile_key(name: str) -> str:
    m = rx.search(name)
    if not m:
        return None
    return f"x{m.group('x')}_y{m.group('y')}"

mosaics = {tile_key(p.name): p for p in LOCAL_MOSAIC.glob("*.tif")}
gts     = {tile_key(p.name): p for p in LOCAL_GT.glob("*.tif")}

keys_m = set(mosaics.keys())
keys_g = set(gts.keys())

missing_gt = sorted(list(keys_m - keys_g))
missing_mo = sorted(list(keys_g - keys_m))

print("keys mosaic:", len(keys_m), "keys gt:", len(keys_g))
print("missing gt:", len(missing_gt))
print("missing mosaic:", len(missing_mo))

assert len(missing_gt) == 0 and len(missing_mo) == 0, "tem tile faltando entre mosaic e gt"

manifest = pd.DataFrame([{
    "key": k,
    "mosaic_path": str(mosaics[k]),
    "gt_path": str(gts[k]),
} for k in sorted(keys_m)])

manifest_path = OUT_LOGS / "manifest_tiles.csv"
manifest.to_csv(manifest_path, index=False)
print("manifest salvo:", manifest_path)
manifest.head()

keys mosaic: 169 keys gt: 169
missing gt: 0
missing mosaic: 0
manifest salvo: /content/drive/MyDrive/unet_dataset_mt2023_v2/logs/manifest_tiles.csv


,key,mosaic_path,gt_path
0,x-50p00_y-10p00,/content/gee_exports_local/mosaic/unet_mt_2023...,/content/gee_exports_local/gt/unet_mt_2023_gtv...
1,x-50p00_y-11p00,/content/gee_exports_local/mosaic/unet_mt_2023...,/content/gee_exports_local/gt/unet_mt_2023_gtv...
2,x-50p00_y-12p00,/content/gee_exports_local/mosaic/unet_mt_2023...,/content/gee_exports_local/gt/unet_mt_2023_gtv...
3,x-50p00_y-13p00,/content/gee_exports_local/mosaic/unet_mt_2023...,/content/gee_exports_local/gt/unet_mt_2023_gtv...
4,x-50p00_y-14p00,/content/gee_exports_local/mosaic/unet_mt_2023...,/content/gee_exports_local/gt/unet_mt_2023_gtv...


In [10]:
# CÉLULA 7 — valida alinhamento (crs, transform, formato) em amostra + full checagem leve

def ds_sig(path: str):
    with rasterio.open(path) as ds:
        return {
            "crs": str(ds.crs),
            "transform": tuple(ds.transform),
            "width": ds.width,
            "height": ds.height,
            "count": ds.count,
            "dtype": ds.dtypes[0],
        }

# amostra rápida
sample = manifest.sample(5, random_state=42)
for _, r in sample.iterrows():
    sm = ds_sig(r["mosaic_path"])
    sg = ds_sig(r["gt_path"])
    ok = (sm["crs"] == sg["crs"]) and (sm["transform"] == sg["transform"]) and (sm["width"] == sg["width"]) and (sm["height"] == sg["height"])
    print(r["key"], "ok=" , ok)

# full checagem leve: abre só metadados
bad = []
for _, r in tqdm(manifest.iterrows(), total=len(manifest), desc="full meta check"):
    sm = ds_sig(r["mosaic_path"])
    sg = ds_sig(r["gt_path"])
    if not ((sm["crs"] == sg["crs"]) and (sm["transform"] == sg["transform"]) and (sm["width"] == sg["width"]) and (sm["height"] == sg["height"])):
        bad.append(r["key"])

print("bad tiles:", len(bad))
assert len(bad) == 0, f"alinhamento falhou em {len(bad)} tiles"
print("ALINHAMENTO OK ✅")

x-60p00_y-18p00 ok= True
x-52p00_y-14p00 ok= True
x-59p00_y-12p00 ok= True
x-52p00_y-13p00 ok= True
x-61p00_y-10p00 ok= True


full meta check: 100%|██████████| 169/169 [00:00<00:00, 187.46it/s]

bad tiles: 0
ALINHAMENTO OK ✅


In [11]:
# CÉLULA 8 — parâmetros de patches (ajuste aqui)

PATCH = 256
STRIDE = 256         # se quiser mais patches (mais overlap), use 128
SCALE_U16 = 10000.0  # seu export do mosaico foi u16 = float*10000

# mínimos de pixels para aceitar patch
MIN_POS_PIX = 200    # milho suficiente
MIN_NEG_PIX = 200    # não-milho suficiente (e npos==0)

# alvo final (p/ começar enxuto e não explodir disco/drive)
TARGET = {"train": 3000, "val": 600, "test": 600}

# shards pequenos = menos ram + menos risco
SHARD_SIZE = 32      # patches por arquivo .npz

# rng
SEED = 42

print("PATCH:", PATCH, "STRIDE:", STRIDE, "TARGET:", TARGET, "SHARD_SIZE:", SHARD_SIZE)

PATCH: 256 STRIDE: 256 TARGET: {'train': 3000, 'val': 600, 'test': 600} SHARD_SIZE: 32


In [12]:
# CÉLULA 9 — fun??es auxiliares (sem carregar mosaico inteiro em RAM)

def read_gt_full(gt_path: str) -> np.ndarray:
    # gt tem 3 bandas: gt_train, gt_test, gt_val (byte)
    with rasterio.open(gt_path) as ds:
        arr = ds.read()  # (3, H, W)
    return arr

def band_index_for_split(split: str) -> int:
    # export: band1=gt_train, band2=gt_test, band3=gt_val
    if split == "train":
        return 0
    if split == "test":
        return 1
    if split == "val":
        return 2
    raise ValueError(split)

def integral_count(mask: np.ndarray) -> np.ndarray:
    # m?scara: H,W bool -> integral H+1,W+1 int64
    m = mask.astype(np.int64, copy=False)
    ii = np.zeros((m.shape[0] + 1, m.shape[1] + 1), dtype=np.int64)
    ii[1:, 1:] = np.cumsum(np.cumsum(m, axis=0), axis=1)
    return ii

def rect_sum(ii: np.ndarray, x0: int, y0: int, x1: int, y1: int) -> int:
    # ii formato (H+1,W+1), coords em pixels
    A = ii[y1, x1]
    B = ii[y0, x1]
    C = ii[y1, x0]
    D = ii[y0, x0]
    return int(A - B - C + D)

def read_mosaic_window(mosaic_ds, x0: int, y0: int, patch: int) -> np.ndarray:
    # lê janela (bands, patch, patch) uint16
    w = Window(x0, y0, patch, patch)
    X = mosaic_ds.read(window=w)  # (B,H,W)
    # -> (H,W,B)
    X = np.transpose(X, (1,2,0))
    return X

def write_shard(split: str, shard_idx: int, X_list, Y_list, meta_list) -> Path:
    out_dir = OUT_SHARDS / split
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"shard_{shard_idx:04d}.npz"

    X = np.stack(X_list, axis=0)  # (N,H,W,C) uint16
    Y = np.stack(Y_list, axis=0)  # (N,H,W) uint8
    meta = np.array(meta_list, dtype=object)

    # salva direto no drive (persistente)
    np.savez_compressed(out_path, X=X, Y=Y, meta=meta)
    return out_path

In [13]:
# CÉLULA 10 — extração de patches (balanceado) sem estourar RAM

rng = np.random.default_rng(SEED)

# opcional: embaralha tiles pra não concentrar região
tile_order = manifest.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

stats = {s: {"kept":0, "pos":0, "neg":0, "shards":0} for s in TARGET.keys()}

for split in ["train","val","test"]:
    target = TARGET[split]
    target_pos = target // 2
    target_neg = target - target_pos

    shard_idx = 0
    X_buf, Y_buf, meta_buf = [], [], []

    pbar = tqdm(total=target, desc=f"[{split}]")

    for _, r in tile_order.iterrows():
        if stats[split]["kept"] >= target:
            break

        mosaic_path = r["mosaic_path"]
        gt_path = r["gt_path"]

        # lê gt inteiro (leve) e prepara integral
        gt3 = read_gt_full(gt_path)                   # (3,H,W) uint8
        M = gt3[band_index_for_split(split)]          # (H,W)
        pos_mask = (M == 1)
        neg_mask = (M == 0)

        # se tile não tem informação p/ esse divis?o, pula rápido
        if pos_mask.sum() == 0 and neg_mask.sum() == 0:
            continue

        II_pos = integral_count(pos_mask)
        II_neg = integral_count(neg_mask)
        H, W = M.shape

        if H < PATCH or W < PATCH:
            continue

        # abre mosaic uma vez e amostra várias janelas
        with rasterio.open(mosaic_path) as mos:
            tries = 0
            max_tries = 4000  # por tile

            while stats[split]["kept"] < target and tries < max_tries:
                tries += 1

                need_pos = stats[split]["pos"] < target_pos
                need_neg = stats[split]["neg"] < target_neg
                if not (need_pos or need_neg):
                    break

                x0 = int(rng.integers(0, W - PATCH + 1))
                y0 = int(rng.integers(0, H - PATCH + 1))
                x1, y1 = x0 + PATCH, y0 + PATCH

                npos = rect_sum(II_pos, x0, y0, x1, y1)
                nneg = rect_sum(II_neg, x0, y0, x1, y1)

                want_pos = need_pos and (npos >= MIN_POS_PIX)
                want_neg = need_neg and (npos == 0) and (nneg >= MIN_NEG_PIX)

                if not (want_pos or want_neg):
                    continue

                # lê só a janela do mosaico (não explode ram)
                Xw = read_mosaic_window(mos, x0, y0, PATCH)      # (H,W,C) uint16
                Yw = M[y0:y1, x0:x1].astype(np.uint8, copy=False)

                kind = "pos" if want_pos else "neg"
                X_buf.append(Xw)
                Y_buf.append(Yw)
                meta_buf.append({
                    "tile": Path(mosaic_path).name,
                    "key": r["key"],
                    "split": split,
                    "kind": kind,
                    "x0": x0, "y0": y0
                })

                stats[split]["kept"] += 1
                stats[split][kind] += 1
                pbar.update(1)

                # salva shard
                if len(X_buf) >= SHARD_SIZE:
                    outp = write_shard(split, shard_idx, X_buf, Y_buf, meta_buf)
                    stats[split]["shards"] += 1
                    shard_idx += 1
                    X_buf, Y_buf, meta_buf = [], [], []
                    # libera ram
                    gc.collect()

                if stats[split]["kept"] % 400 == 0:
                    free_gb = shutil.disk_usage("/content").free / (1024**3)
                    print(f"[{split}] kept={stats[split]['kept']}/{target} pos={stats[split]['pos']} neg={stats[split]['neg']} free_gb={free_gb:.2f}")

        # libera gt do tile
        del gt3, M, pos_mask, neg_mask, II_pos, II_neg
        gc.collect()

    # flush final
    if X_buf:
        outp = write_shard(split, shard_idx, X_buf, Y_buf, meta_buf)
        stats[split]["shards"] += 1

    pbar.close()

print("DONE ✅")
print(json.dumps(stats, indent=2))

(OUT_LOGS / "patch_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
print("stats salvo em:", OUT_LOGS / "patch_stats.json")

[train]:  14%|█▍        | 414/3000 [01:45<05:07,  8.42it/s]

[train] kept=400/3000 pos=207 neg=193 free_gb=22.96


[train]:  27%|██▋       | 801/3000 [03:27<09:12,  3.98it/s]

[train] kept=800/3000 pos=414 neg=386 free_gb=23.63


[train]:  40%|███▉      | 1198/3000 [05:05<07:44,  3.88it/s]

...

[val] kept=400/600 pos=156 neg=244 free_gb=22.95


[test]:  67%|██████▋   | 401/600 [01:41<00:41,  4.82it/s]

[test] kept=400/600 pos=272 neg=128 free_gb=22.95


[test]: 100%|██████████| 600/600 [02:40<00:00,  3.74it/s]

DONE ✅
{
  "train": {
    "kept": 3000,
    "pos": 1500,
    "neg": 1500,
    "shards": 94
  },
  "val": {
    "kept": 600,
    "pos": 300,
    "neg": 300,
    "shards": 19
  },
  "test": {
    "kept": 600,
    "pos": 300,
    "neg": 300,
    "shards": 19
  }
}
stats salvo em: /content/drive/MyDrive/unet_dataset_mt2023_v2/logs/patch_stats.json


In [14]:
# CÉLULA 11 — checagem r?pida: abre 1 shard e valida shapes/valores

# pega 1 shard de treino
train_shards = sorted((OUT_SHARDS/"train").glob("shard_*.npz"))
assert len(train_shards) > 0, "não achei shards em train"

p = train_shards[0]
z = np.load(p, allow_pickle=True)
X = z["X"]  # uint16 (N,H,W,C)
Y = z["Y"]  # uint8  (N,H,W)

print("shard:", p.name)
print("X:", X.shape, X.dtype, "min/max:", int(X.min()), int(X.max()))
print("Y:", Y.shape, Y.dtype, "unique:", np.unique(Y)[:10], "...")
print("meta example:", z["meta"][0])

shard: shard_0000.npz
X: (32, 256, 256, 21) uint16 min/max: 0 10000
Y: (32, 256, 256) uint8 unique: [  0 255] ...
meta example: {'tile': 'unet_mt_2023_mosaic_x-59p00_y-12p00.tif', 'key': 'x-59p00_y-12p00', 'split': 'train', 'kind': 'neg', 'x0': 2488, 'y0': 3022}


In [ ]:
# CÉLULA 12 — limpeza opcional (para liberar /content depois que shards estiverem no drive)

# cuidado: só rode quando confirmar que os shards no drive existem
print("local mosaic:", len(list(LOCAL_MOSAIC.glob('*.tif'))))
print("local gt:", len(list(LOCAL_GT.glob('*.tif'))))

# apaga local (não apaga o drive)
# shutil.rmtree(LOCAL_ROOT)
# print("apagou local:", LOCAL_ROOT)

local mosaic: 169
local gt: 169
